# Module 5 — Weighting: IPW (Inverse Probability Weighting) and Doubly Robust

Part of the Causal Inference Playbook. Program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

- The running question: did NSW (National Supported Work), a US job-training program for men with very low earnings, raise the earnings of the men who took it?
- The men who took the program are called the treated men. They're compared with men drawn from the PSID (Panel Study of Income Dynamics), a US household survey, who didn't take it: the comparison men. Each man has "before" columns: facts about him from before the program, such as his age, his education and his earlier earnings. The outcome is his earnings after the program.
- The treated men were different from the comparison men even before the program started: for example, they earned far less. So a plain comparison of the two groups mixes the program's effect with those differences.
- Module 5 uses weighting: every comparison man stays in, but each counts more or less, so that the comparison group as a whole looks like the treated men before the program. Comparison men like the treated men count a lot, and men unlike them count almost nothing.
- What can go wrong: when only a few comparison men are like the treated men, those few get huge weights, and the estimate rests on them.

**What this module produces:**
1. A weighting estimate of the program's effect on the real data, a check of whether the weighted comparison men look like the treated men, and the estimate's uncertainty.
2. The doubly robust estimate. It combines weighting with regression adjustment, which predicts each treated man's earnings without the program from a formula fitted on the comparison men. It also shows whether that formula's predictions are off for men like the treated men.

## From the curriculum

> ## <u>Module 5 — Weighting: IPW (Inverse Probability Weighting) and Doubly Robust</u>
>
> **Concepts**
> - A man's propensity score, as in Module 4: take the men, treated and
>   comparison, whose "before" columns are like his. His score is the share
>   of treated men among them.
> - IPW (inverse probability weighting) for the ATT (average treatment effect
>   on the treated): every comparison man stays in, with a weight, (his score)
>   ÷ (1 − his score): the number of treated men he stands in for. Each
>   treated man counts once.
> - The weighted mean of the comparison men's 1978 earnings (after the
>   program) stands in for the
>   treated men's mean without the program.
> - Balance check: after weighting, compare the mean of every "before" column
>   for the treated men and for the weighted comparison men. Also, how much of
>   the weight a few men carry.
> - Doubly robust uses two formulas:
>   1. The score formula: the fitted formula that gives each man's
>      propensity score.
>   2. The earnings formula: regression adjustment's straight-line formula,
>      fitted on the comparison men, which predicts each treated man's 1978
>      earnings without the program (Module 3).
>
>   The doubly robust estimate is regression adjustment's estimate minus
>   the earnings formula's weighted mean miss (actual − predicted earnings)
>   on the comparison men. It gets the right answer, apart from chance, if
>   either formula is right.
>
> **Why it matters**
> Weighting keeps every comparison man. Unlike matching (Module 4), it
> doesn't pair each treated man with one closest comparison man. It also
> gives almost no weight to the comparison men unlike every treated man,
> without a cut-off chosen by hand (such as "drop every comparison man below a
> given score"). Doubly robust is right if
> either of its two formulas is right, and its correction shows when the
> earnings formula is wrong.
>
> **Worked example**
> 1. NSW (National Supported Work) job-training data vs the PSID (Panel Study
>    of Income Dynamics) men: the weights, where they go, the balance check
>    and the effect estimate, with a 95% confidence interval.
> 2. Doubly robust on the same data, next to regression adjustment and
>    weighting.
>
> **Resources**
> - [Causal Inference: The Mixtape, ch. 5 "Unconfoundedness"](https://mixtape.scunning.com/05-causal_cross_sectional_designs)
>   (Scott Cunningham, ~30 min for section 5.8 "Estimation with Inverse
>   Probability Weighting", verified): IPW on the same NSW data.
> - [The Effect, ch. 14 "Matching"](https://theeffectbook.net/ch-Matching.html)
>   (Nick Huntington-Klein, ~40 min for sections 14.5 and 14.7.2, verified):
>   propensity score weighting, and doubly robust estimation.
>
> **Exercise type:** redo weighting and doubly robust with the CPS (Current
> Population Survey, a monthly US labour survey) comparison group. Fully
> worked.
>
> **Interview angle:** "Users who turned on feature X spend 10% more. We
> weighted the users without X to look like the users with X, using a
> propensity score, and the gap is still 10%. Is that the effect of X?"

## Lesson

### The data

- The treated men: 185 men who took NSW in 1976 or 1977. They had applied, and a lottery let them in.
- The comparison men: 2,490 men drawn from the PSID, who didn't apply to the program, so didn't take it.
- The 8 "before" columns: age, years of education, black, hispanic, married, no degree (no high-school degree), 1974 earnings and 1975 earnings. Black, hispanic, married and no degree are yes/no columns: 1 for yes, 0 for no.
- The outcome: 1978 earnings. All earnings are in US dollars per year.
- The benchmark: the same lottery kept 260 applicants out. The lottery, not the men, decided who got in, so the 185 treated men and these 260 men differ only by chance. So the 260 men's mean 1978 earnings shows what the treated men would have earned without the program. The treated men's mean minus the 260 men's mean is the program's effect on the treated men, with no selection bias (the part of a gap that comes from the groups being different to begin with).
- The naive gap: the plain comparison of the treated men with the PSID men. It mixes the program's effect with how different the two groups were.

**The motive:** the benchmark, the same one used in Modules 1–4, is the answer each method below should get close to. The naive gap is the unadjusted comparison each method tries to improve on.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

import common
from common import show

# load_lalonde loads the NSW data, named after the economist LaLonde, who first used it to test these methods
exp = common.load_lalonde("experimental")  # the lottery: 185 treated men (treat = 1) and the 260 men kept out (treat = 0)
psid = common.load_lalonde("psid")  # the same 185 treated men, plus the 2,490 PSID men (treat = 0)
nsw_t = exp[exp.treat == 1].copy()
lottery = exp[exp.treat == 0]
psid_c = psid[psid.treat == 0].copy()
COLS8 = ["age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75"]

benchmark = nsw_t.re78.mean() - lottery.re78.mean()
print(f"Benchmark = (mean 1978 earnings of the treated men, {len(nsw_t)} men) - (mean 1978 earnings of the men kept out by the lottery, {len(lottery)} men) = "
      f"{nsw_t.re78.mean():,.1f} - {lottery.re78.mean():,.1f} = {benchmark:,.1f}")
naive = nsw_t.re78.mean() - psid_c.re78.mean()
print(f"Naive gap = (mean 1978 earnings of the treated men, {len(nsw_t)} men) - (mean 1978 earnings of the PSID men, {len(psid_c):,} men) = "
      f"{nsw_t.re78.mean():,.1f} - {psid_c.re78.mean():,.1f} = {naive:,.1f}")

Benchmark = (mean 1978 earnings of the treated men, 185 men) - (mean 1978 earnings of the men kept out by the lottery, 260 men) = 6,349.1 - 4,554.8 = 1,794.3
Naive gap = (mean 1978 earnings of the treated men, 185 men) - (mean 1978 earnings of the PSID men, 2,490 men) = 6,349.1 - 21,553.9 = -15,204.8


**Observation:** the naive gap, (−$15,204.8), makes the program look harmful. The benchmark, $1,794.3, says it raised earnings.

### The propensity score

Weighting is built on the propensity score, the same score Module 4 matched on:
- A man's propensity score: take the men whose 8 "before" columns are like his, from all 2,675 men (the 185 treated men and the 2,490 PSID men). His score is the share of treated men among them. A man's score depends only on his columns, not on which group he's actually in, so a PSID man can have a high score. It's a number between 0 and 1.
- Almost no two men have exactly the same 8 columns, so this share can't be counted directly. It's estimated by logistic regression: a regression that predicts a yes/no column (here, 1 for a treated man, 0 for a PSID man) from other columns. Unlike a straight line, its predictions never go below 0 or above 1, so they can be read as shares.
- This fitted logistic regression is called the score formula below.

### The weight

The question is about the treated men: the ATT (average treatment effect on the treated) is how much the program changed their 1978 earnings, on average. The missing number is the treated men's mean 1978 earnings without the program. The PSID men's earnings stand in for it, but only after each PSID man is given a weight: how many treated men he stands in for.

Say a PSID man's score is 0.75: among 100 men with columns like his, 75 are treated men and 25 are PSID men. To stand in for those 75 treated men, each of the 25 PSID men has to count 75 ÷ 25 = 3 times. In the score's own terms:
- Weight of a PSID man = (his score) ÷ (1 − his score). Here, 0.75 ÷ 0.25 = 3.
- A score of 0.1: among 100 men like him, 10 are treated and 90 are PSID men, so his weight = 0.1 ÷ 0.9 = 0.111. He stands in for a ninth of a treated man.
- A score close to 1: almost every man like him is treated, so the few PSID men like him get very large weights.

Each treated man counts once: a weight of 1.

Because each PSID man's weight equals the number of treated men he stands in for, the weighted PSID men should have the same mix on the 8 columns as the treated men, if the scores are accurate. This method is called IPW (inverse probability weighting).

### Weighting in three steps

1. Fit each man's propensity score, and give each PSID man his weight = score ÷ (1 − score).
2. Weighted mean 1978 earnings of the PSID men = (sum of weight × 1978 earnings) ÷ (sum of the weights). It stands in for the treated men's mean 1978 earnings without the program.
3. ATT estimate = (the treated men's mean 1978 earnings) − (the PSID men's weighted mean).

The weights make the groups alike only if the score formula is right: for men with any given 8 columns, its score equals the real share of treated men among them. So this has to be checked on the data, column by column, with the weighted means: the balance check.

### What it assumes

1. Every confounder has a column. A confounder is a trait that affects both who took the program and earnings afterwards. Weighting can only make the groups alike on the columns it sees. A confounder with no column (motivation, for example) isn't weighted for.
2. Overlap: every treated man has PSID men with a similar score. Where only a few PSID men are like the treated men, those few get huge weights, and the estimate swings with their earnings.

## Worked example 1: weighting on the real data

**The motive:** the naive gap, (−$15,204.8), says the program cut earnings by over $15,000, against a benchmark of $1,794.3. If the weights make the PSID men look like the treated men, the estimate should move toward the benchmark.

### Step 1 of 3: each PSID man's weight

**The motive:** each PSID man's 1978 earnings count in proportion to his weight, so the men with the largest weights count most. The table shows, for each score range, how many men are in it and the weights of the PSID men in it. Its last two columns compare the range's share of the treated men with its share of the PSID men's total weight: if the weights work, the two should be close. The lines under the table count how few PSID men carry half the total weight, and how much weight goes to the PSID men with a lower score than every treated man. Those men are unlike every treated man, so they shouldn't count.

In [2]:
SCORE = "treat ~ " + " + ".join(COLS8)
both = pd.concat([nsw_t, psid_c], ignore_index=True)
both["score"] = smf.logit(SCORE, both).fit(disp=0).predict(both)
nsw_t["score"] = both.score[both.treat == 1].to_numpy()
psid_c["score"] = both.score[both.treat == 0].to_numpy()
psid_c["weight"] = psid_c.score / (1 - psid_c.score)

edges = np.linspace(0, 1, 11)
labels = [f"{a:.1f} < score <= {b:.1f}" for a, b in zip(edges[:-1], edges[1:])]
t_bin, c_bin = pd.cut(nsw_t.score, edges, labels=labels), pd.cut(psid_c.score, edges, labels=labels)
total = psid_c.weight.sum()
rows = []
for b in labels:
    w_b = psid_c.weight[c_bin == b]
    rows.append((b, (t_bin == b).sum(), len(w_b), f"{w_b.min():.3f} to {w_b.max():.3f}" if len(w_b) else "—",
                 f"{(t_bin == b).mean():.1%}", f"{w_b.sum() / total:.1%}"))
show(pd.DataFrame(rows, columns=["propensity score", f"treated men ({len(nsw_t)} men)", f"PSID men ({len(psid_c):,} men)",
                                 "weight of a PSID man in this range", "share of the treated men", "share of the PSID men's total weight"]))
print()

by_weight = psid_c.weight.sort_values(ascending=False)
half = (by_weight.cumsum() < total / 2).sum() + 1
below_all = psid_c.score < nsw_t.score.min()
print(f"Largest weight of any PSID man = {by_weight.iloc[0]:.1f}")
print(f"PSID men who together carry half the total weight = {half} of {len(psid_c):,}")
print(f"PSID men with a lower score than every treated man = {below_all.sum():,} of {len(psid_c):,}, carrying {psid_c.weight[below_all].sum() / total:.2%} of the total weight")

propensity score    treated men (185 men)  PSID men (2,490 men)  weight of a PSID man in this range  share of the treated men  share of the PSID men's total weight
0.0 < score <= 0.1                     10                  2326  0.000 to 0.111                                          5.4%                                  8.1%
0.1 < score <= 0.2                      8                    72  0.112 to 0.248                                          4.3%                                  6.8%
0.2 < score <= 0.3                     13                    28  0.250 to 0.423                                          7.0%                                  4.8%
0.3 < score <= 0.4                     11                    20  0.431 to 0.660                                          5.9%                                  5.8%
0.4 < score <= 0.5                     13                    11  0.691 to 0.976                                          7.0%                                  5.0%
0.5 < score <= 0

**Observation:**
- 2,326 of the 2,490 PSID men have a score of 0.1 or less. Each counts as a ninth of a man or less, and together they carry 8.1% of the total weight.
- Above 0.7, there are 28 + 47 + 29 = 104 treated men and only 7 + 8 + 3 = 18 PSID men. Those 18 men carry 12.7% + 28.7% + 16.0% = 57.4% of the total weight.
- 14 of the 2,490 PSID men carry half the total weight. The largest weight is 10.8: one PSID man stands in for 10.8 treated men.
- The 1,330 PSID men with a lower score than every treated man carry 0.02% of the weight: their scores are so low that their weights are close to 0. Weighting leaves them out almost entirely, with no cut-off chosen by hand, such as "drop every PSID man below a given score".

**Conclusion:** the weights put the PSID men's weight roughly where the treated men are, which is what they're for. Above 0.7: 15.1% + 25.4% + 15.7% = 56.2% of the treated men, and 57.4% of the weight. The largest gap in a single range is at 0.8 < score <= 0.9: 28.7% − 25.4% = 3.3 percentage points, so the match isn't exact. Step 2 checks it column by column. But the price is that the estimate leans on a few men: half the total weight sits on 14 PSID men.

### Step 2 of 3: the balance check

**The motive:** if the weights did their job, the weighted PSID men should look like the treated men, on average, on all 8 columns. The table compares the means, with all the PSID men unweighted next to them, to show how different they were before weighting. For a 1/0 column, the mean is the share of men with a 1: for example, the share who are black. Difference after weighting = (the PSID men's weighted mean) − (the treated men's mean). For a 1/0 column, the difference is in percentage points.

To judge whether a difference is big, it's compared with how much the column varies among the treated men:
- Spread = the column's standard deviation among the treated men: the typical distance of a treated man's value from the treated men's mean. The treated men's spread is used because the effect is measured on them. For a 1/0 column, the spread is in percentage points, like the difference.
- Size = (difference after weighting) ÷ (spread). It has no unit, so dollars, years and percentage points can be compared.
- The rule: a column passes if its size, ignoring the sign, is under 0.1. The check passes only if all 8 columns pass. This is the most common balance test, the size test from Module 4. The 0.1 is a convention, not derived from the data.

In [3]:
names = {"age": "age", "education": "years of education", "black": "black", "hispanic": "hispanic", "married": "married",
         "nodegree": "no degree", "re74": "1974 earnings", "re75": "1975 earnings"}
yes_no = {"black", "hispanic", "married", "nodegree"}
fmt = lambda col, v: f"{v:.1%}" if col in yes_no else f"{v:,.1f}"
shown = lambda col, v: round(v, 3) if col in yes_no else round(v, 1)  # differences use the means as printed, so the subtraction checks by hand

def balance(treated, comparison, label):
    rows = []
    for c in COLS8:
        to_points = 100 if c in yes_no else 1  # 1/0 columns in percentage points
        weighted = (comparison.weight * comparison[c]).sum() / comparison.weight.sum()
        after = round((shown(c, weighted) - shown(c, treated[c].mean())) * to_points, 1)
        spread = round(treated[c].std() * to_points, 1)
        size = after / spread
        rows.append((names[c], "1/0" if c in yes_no else "number", fmt(c, treated[c].mean()), fmt(c, comparison[c].mean()), fmt(c, weighted),
                     f"{after + 0:,.1f}", f"{spread:,.1f}", f"{size + 0:.3f}", "yes" if abs(size) < 0.1 else "no"))
    return pd.DataFrame(rows, columns=["column", "type", f"treated men ({len(treated)} men): mean", f"all {label} men ({len(comparison):,} men): mean",
                                       f"{label} men, weighted: mean", "difference after weighting", f"spread among the treated men ({len(treated)} men)",
                                       "size = difference ÷ spread", "passes (size under 0.1)?"])

show(balance(nsw_t, psid_c, "PSID"))
edu_gap = (psid_c.weight * psid_c.education).sum() / psid_c.weight.sum() - nsw_t.education.mean()
print(f"Years of education: the 0.0 is rounded. Unrounded difference = {edu_gap:.3f} years")

column              type    treated men (185 men): mean  all PSID men (2,490 men): mean  PSID men, weighted: mean  difference after weighting  spread among the treated men (185 men)  size = difference ÷ spread  passes (size under 0.1)?
age                 number                         25.8                            34.9                      25.2                        -0.6                                     7.2                      -0.083  yes
years of education  number                         10.3                            12.1                      10.3                         0.0                                     2.0                       0.000  yes
black               1/0                           84.3%                           25.1%                     85.2%                         0.9                                    36.5                       0.025  yes
hispanic            1/0                            5.9%                            3.3%                      6.5%      

**Observation:** 7 of the 8 columns pass. 1974 earnings fails, just: size 0.102. So the balance check fails, on one column. The weighted PSID men earned $497.1 more in 1974 than the treated men.

**Conclusion:** weighting closes almost all of the differences between the groups, but not all. For example, 1974 earnings: all the PSID men's mean was $19,428.7 against the treated men's $2,095.6, and the weighted mean is $2,592.7. By size, the largest left are 1974 earnings (0.102), age (−0.083) and married (−0.074).

**Hypothesis:** if these leftover differences affect 1978 earnings, the weighted PSID men's mean 1978 earnings is a bit off, and so is the estimate in step 3. Worked example 2 measures by how much.

### Step 3 of 3: the ATT estimate, and how sure it is

**The motive:** with the weights set, the ATT estimate is the treated men's mean minus the PSID men's weighted mean. Then the 95% confidence interval shows how much it could move if the study were rerun on another sample of treated men and PSID men.

The interval comes from the bootstrap, as in Module 3: draw 185 treated men from the 185, and separately 2,490 PSID men from the 2,490, at random, with replacement. Do it 1,000 times. Each time, refit the score formula, recompute the weights and redo the estimate. The middle 95% of the 1,000 estimates is the interval.

Why the bootstrap here, when Module 4 used a formula for matching's interval:

1. The bootstrap gives a correct interval only when a slightly different sample gives a slightly different estimate.
2. Matching fails this test. In a redrawn sample, a treated man's closest PSID man may be left out. He is then compared with a different man, with different 1978 earnings, and the estimate jumps. So Module 4 used a formula for matching's interval, and the bootstrap only for regression's.
3. Weighting only partly passes it. It has no switches: no man is swapped for another. But half the total weight sits on 14 PSID men (step 1), so a redrawn sample that leaves out a few of them, or draws them twice, can move the estimate a lot. So weighting's interval is only a rough guide. This module still uses the bootstrap, the usual tool for weighting, for all three of its intervals: weighting's (here), regression's (below) and doubly robust's (below). It does so because no method gives an exact interval here: a formula for weighting's interval is thrown off by the same 14 men.

In [4]:
def weighting_att(treated, comparison):
    # The weighted mean of the comparison men's 1978 earnings, and the ATT estimate.
    weighted = (comparison.weight * comparison.re78).sum() / comparison.weight.sum()
    return weighted, treated.re78.mean() - weighted

add_one = lambda d: np.column_stack([np.ones(len(d)), d[COLS8].to_numpy(float)])

def bootstrap(comparison, estimate):
    # Redraw the comparison men and the treated men with replacement, refit the score, redo the estimate. Returns the middle 95%.
    X_c, y_c, X_t, y_t = add_one(comparison), comparison.re78.to_numpy(float), add_one(nsw_t), nsw_t.re78.to_numpy(float)
    rng = np.random.default_rng(3)
    boot = []
    for _ in range(1000):
        i_c, i_t = rng.integers(0, len(y_c), len(y_c)), rng.integers(0, len(y_t), len(y_t))
        coef = sm.Logit(np.r_[np.ones(len(i_t)), np.zeros(len(i_c))], np.vstack([X_t[i_t], X_c[i_c]])).fit(disp=0).params
        score = 1 / (1 + np.exp(-(X_c[i_c] @ coef)))  # the score formula's prediction for each PSID man
        boot.append(estimate(X_c[i_c], y_c[i_c], X_t[i_t], y_t[i_t], score / (1 - score)))
    low, high = np.percentile(boot, [2.5, 97.5])
    return f"{low:,.1f} to {high:,.1f}"

ipw = lambda X_c, y_c, X_t, y_t, w: y_t.mean() - (w * y_c).sum() / w.sum()

r1 = lambda v: round(float(v), 1)  # every printed result below is computed from the printed (rounded) numbers, so each line checks by hand
weighted_psid, att_ipw = weighting_att(nsw_t, psid_c)
sum_wy, sum_w = r1((psid_c.weight * psid_c.re78).sum()), round(psid_c.weight.sum(), 3)  # 3 decimals, so the division checks by hand to 0.1
print(f"Weighted mean 1978 earnings, PSID men ({len(psid_c):,} men) = (sum of weight × 1978 earnings) ÷ (sum of the weights) = "
      f"{sum_wy:,.1f} ÷ {sum_w:,.3f} = {r1(sum_wy / sum_w):,.1f}")
print(f"ATT estimate = (mean 1978 earnings, treated men, {len(nsw_t)} men) - (weighted mean 1978 earnings, PSID men) = "
      f"{nsw_t.re78.mean():,.1f} - {r1(sum_wy / sum_w):,.1f} = {r1(nsw_t.re78.mean()) - r1(sum_wy / sum_w):,.1f}")
ipw_interval = bootstrap(psid_c, ipw)
print(f"95% confidence interval (bootstrap, 1,000 draws) = {ipw_interval}")

Weighted mean 1978 earnings, PSID men (2,490 men) = (sum of weight × 1978 earnings) ÷ (sum of the weights) = 845,555.5 ÷ 184.205 = 4,590.3
ATT estimate = (mean 1978 earnings, treated men, 185 men) - (weighted mean 1978 earnings, PSID men) = 6,349.1 - 4,590.3 = 1,758.8


95% confidence interval (bootstrap, 1,000 draws) = -153.3 to 3,540.9


**Observation:**
- Weighting moves the estimate from the naive gap, (−$15,204.8), to $1,758.8. Gap to the benchmark = 1,794.3 − 1,758.8 = $35.5: the estimate is $35.5 below it.
- The 95% confidence interval runs from (−$153.3) to $3,540.9. It contains 0: the estimate can't rule out, at the 95% level, that the program did nothing.


**Conclusion:** weighting lands $35.5 from the benchmark, but the interval is 3,540.9 − (−153.3) = $3,694.2 wide, so landing this close doesn't show that weighting is this accurate. Half the total weight sits on 14 PSID men (step 1), so a redrawn sample that leaves out a few of them, or draws them twice, can move the estimate a lot.

## Worked example 2: doubly robust

### Regression adjustment's weak point

Regression adjustment (Module 3):
1. Fit the earnings formula on the PSID men: predicted 1978 earnings = (starting point) + the sum, over the 8 columns, of (the column's coefficient × the man's value).
2. Use the earnings formula to predict each treated man's 1978 earnings without the program.
3. Regression adjustment's ATT = the mean of (actual − predicted), over the treated men.

Miss on a man = (the man's actual 1978 earnings) − (the earnings formula's prediction for the man).

Weak point: the earnings formula is fitted on all 2,490 PSID men, most of them unlike the treated men. So the earnings formula may predict badly for the few PSID men like the treated men. If the earnings formula predicts too high for men like the treated men, regression adjustment's ATT comes out too low, and nothing in regression adjustment shows it.

**The motive:** the doubly robust ATT uses weighting's weights to measure how far off the earnings formula is for men like the treated men, and corrects regression adjustment's ATT by that amount.

### The doubly robust estimate

The treated men's earnings without the program are unseen, so the earnings formula's miss on them is unknown. The weighted PSID men stand in: their earnings are known, and after the weighting they nearly match the treated men (step 2: only 1974 earnings failed, just).
- Weighted mean miss = (sum of weight × miss) ÷ (sum of the weights).
- Unweighted mean miss = 0.0 always, because the earnings formula's starting point is fitted so that the earnings formula's mean prediction equals the mean actual earnings of the 2,490 PSID men. So the weighted mean miss comes from the weighting alone.
- Doubly robust ATT = (regression adjustment's ATT) − (weighted mean miss).
- A negative weighted mean miss means regression adjustment predicts the treated men's 1978 earnings without the program too high.
- Regression adjustment's ATT = (treated men's actual − treated men's predicted earnings), so that ATT comes out too low.
- Doubly robust raises it by subtracting the negative weighted mean miss: (regression adjustment's ATT) − (weighted mean miss).

The doubly robust ATT is called that, because it uses two formulas, and it is right - if either one is right:
1. the earnings formula, which gives regression adjustment's predictions
2. the score formula, which gives weighting's scores

The proof is in the appendix at the end.

In [5]:
LINEAR = "re78 ~ " + " + ".join(COLS8)
earnings_formula = smf.ols(LINEAR, psid_c).fit()
att_reg = (nsw_t.re78 - earnings_formula.predict(nsw_t)).mean()
miss = psid_c.re78 - earnings_formula.predict(psid_c)
weighted_miss = (psid_c.weight * miss).sum() / psid_c.weight.sum()

predicted_t = r1(earnings_formula.predict(nsw_t).mean())
print("Regression adjustment: ATT estimate")
print(f"    = (mean actual 1978 earnings, treated men, {len(nsw_t)} men) - (mean predicted 1978 earnings without the program, treated men)")
print(f"    = {nsw_t.re78.mean():,.1f} - {predicted_t:,.1f} = {r1(nsw_t.re78.mean()) - predicted_t:,.1f}")
print()
print(f"Weighted mean miss of the earnings formula, PSID men ({len(psid_c):,} men)")
print("    = (sum of weight × miss) ÷ (sum of the weights)")
print(f"    = {(psid_c.weight * miss).sum():,.1f} ÷ {sum_w:,.3f} = {weighted_miss:,.1f}")
print()
print(f"For comparison, unweighted mean miss, PSID men = {round(miss.mean(), 1) + 0:,.1f}")
print()
print("Doubly robust ATT estimate")
print("    = (regression adjustment's estimate) - (weighted mean miss)")
print(f"    = {att_reg:,.1f} - ({weighted_miss:,.1f}) = {r1(att_reg) - r1(weighted_miss):,.1f}")

Regression adjustment: ATT estimate
    = (mean actual 1978 earnings, treated men, 185 men) - (mean predicted 1978 earnings without the program, treated men)
    = 6,349.1 - 5,558.6 = 790.5

Weighted mean miss of the earnings formula, PSID men (2,490 men)
    = (sum of weight × miss) ÷ (sum of the weights)
    = -232,522.5 ÷ 184.205 = -1,262.3

For comparison, unweighted mean miss, PSID men = 0.0

Doubly robust ATT estimate
    = (regression adjustment's estimate) - (weighted mean miss)
    = 790.5 - (-1,262.3) = 2,052.8


**Observation:** weighted to look like the treated men, the PSID men's mean miss is (−$1,262.3): for PSID men like the treated men, the earnings formula predicts 1978 earnings $1,262.3 too high.

**Conclusion:** the weights failed the balance check on one column, so this conclusion rests on the weights being close to right. If the weights are close to right, the earnings formula predicts too high for men like the treated men, so regression adjustment's ATT, $790.5, is too low. The doubly robust ATT adds the $1,262.3 back.

### Comparing the three ATTs

The table puts the three ATT estimates side by side on the same 2,490 PSID men: which lands nearest the benchmark, and how sure each is. Each has a bootstrap 95% confidence interval from the same 1,000 redrawn samples.

In [6]:
def regression(X_c, y_c, X_t, y_t, w):
    coef = np.linalg.lstsq(X_c, y_c, rcond=None)[0]
    return (y_t - X_t @ coef).mean()

def doubly_robust(X_c, y_c, X_t, y_t, w):
    coef = np.linalg.lstsq(X_c, y_c, rcond=None)[0]
    return (y_t - X_t @ coef).mean() - (w * (y_c - X_c @ coef)).sum() / w.sum()

att_dr = r1(att_reg) - r1(weighted_miss)
rows = [
    ("regression adjustment", att_reg, bootstrap(psid_c, regression)),
    ("weighting (IPW)", r1(nsw_t.re78.mean()) - r1(sum_wy / sum_w), ipw_interval),
    ("doubly robust", att_dr, bootstrap(psid_c, doubly_robust)),
]
show(pd.DataFrame({
    "method": [r[0] for r in rows],
    "ATT estimate": [f"{r[1]:,.1f}" for r in rows],
    "95% confidence interval": [r[2] for r in rows],
    f"gap to the benchmark ({benchmark:,.1f} - ATT estimate)": [f"{r1(benchmark) - r1(r[1]):,.1f}" for r in rows],
}))

method                 ATT estimate  95% confidence interval  gap to the benchmark (1,794.3 - ATT estimate)
regression adjustment         790.5  -693.0 to 2,340.1                                              1,003.8
weighting (IPW)             1,758.8  -153.3 to 3,540.9                                                 35.5
doubly robust               2,052.8  311.1 to 3,783.3                                                -258.5


**Observation:**
- Gap to the benchmark, $1,794.3: doubly robust lands $258.5 above, weighting $35.5 below, and regression adjustment $1,003.8 below.
- Doubly robust's interval, $311.1 to $3,783.3, doesn't contain 0. Weighting's interval, (−$153.3) to $3,540.9, and regression adjustment's interval, (−$693.0) to $2,340.1, do.

**Conclusion:**
- The doubly robust correction moved regression adjustment's estimate from $1,003.8 below the benchmark to $258.5 above the benchmark.
- Weighting's estimate is nearest the benchmark here. But every interval is over $3,000 wide (the narrowest, regression adjustment's: 2,340.1 − (−693.0) = $3,033.1), so this one dataset, the 185 treated men and the 2,490 PSID men, can't show that weighting is the more accurate method.
- Did the program raise earnings? All three estimates are positive, but only doubly robust's interval leaves out 0. So the data leans toward a positive effect, but not every method rules out 0 at the 95% level.
- Doubly robust also checks the earnings formula. A right earnings formula gives a weighted mean miss near 0. Here the weighted mean miss, (−$1,262.3), is larger in size than regression adjustment's whole estimate, $790.5. If the weights are close to right, the earnings formula is far off for men like the treated men.

### Why doubly robust and weighting differ: $294.0

Doubly robust's estimate, $2,052.8, is $294.0 above weighting's estimate, $1,758.8. Writing the doubly robust formula out in full, then reordering the terms, shows where the $294.0 comes from. Here "actual" is 1978 earnings, and "predicted" is the earnings formula's prediction of 1978 earnings without the program. For the treated men, "predicted" stands in for the treated men's unseen 1978 earnings without the program.
- Doubly robust ATT = (regression adjustment's ATT) − (weighted mean miss) =<br>
  = (treated men's mean actual − treated men's mean predicted) − (PSID men's weighted mean actual − PSID men's weighted mean predicted).
- Reordered = (treated men's mean actual − PSID men's weighted mean actual) + (PSID men's weighted mean predicted − treated men's mean predicted).
- The first bracket is weighting's estimate. The second bracket is the $294.0.

The second bracket compares the earnings formula's mean prediction for two groups: the weighted PSID men and the treated men. The earnings formula is a straight line, so a group's mean prediction depends only on the group's mean on each of the 8 columns. So the second bracket is 0 only if the weighted PSID men match the treated men's mean on every column. Step 2 showed that the means don't match exactly.

In [7]:
predicted_c = r1((psid_c.weight * earnings_formula.predict(psid_c)).sum() / psid_c.weight.sum())
sum_wp = (psid_c.weight * earnings_formula.predict(psid_c)).sum()
print(f"Weighted mean predicted 1978 earnings, PSID men ({len(psid_c):,} men)")
print("    = (sum of weight × predicted) ÷ (sum of the weights)")
print(f"    = {sum_wp:,.1f} ÷ {sum_w:,.3f} = {predicted_c:,.1f}")
print()
sum_pt = earnings_formula.predict(nsw_t).sum()
print(f"Mean predicted 1978 earnings without the program, treated men ({len(nsw_t)} men)")
print("    = (sum of predicted) ÷ (number of treated men)")
print(f"    = {sum_pt:,.1f} ÷ {len(nsw_t)} = {predicted_t:,.1f}")
print()
dr_r = r1(att_reg) - r1(weighted_miss)
ipw_r = r1(nsw_t.re78.mean()) - r1(sum_wy / sum_w)
print("Gap between doubly robust and weighting")
print("    = (weighted mean predicted 1978 earnings, PSID men) - (mean predicted, treated men)")
print(f"    = {predicted_c:,.1f} - {predicted_t:,.1f} = {predicted_c - predicted_t:,.1f}")
print(f"    check: (doubly robust's estimate) - (weighting's estimate) = {dr_r:,.1f} - {ipw_r:,.1f} = {dr_r - ipw_r:,.1f}")

Weighted mean predicted 1978 earnings, PSID men (2,490 men)
    = (sum of weight × predicted) ÷ (sum of the weights)
    = 1,078,078.0 ÷ 184.205 = 5,852.6

Mean predicted 1978 earnings without the program, treated men (185 men)
    = (sum of predicted) ÷ (number of treated men)
    = 1,028,340.7 ÷ 185 = 5,558.6

Gap between doubly robust and weighting
    = (weighted mean predicted 1978 earnings, PSID men) - (mean predicted, treated men)
    = 5,852.6 - 5,558.6 = 294.0
    check: (doubly robust's estimate) - (weighting's estimate) = 2,052.8 - 1,758.8 = 294.0


**Observation:** second bracket = 5,852.6 − 5,558.6 = $294.0. The earnings formula predicts $294.0 more for the weighted PSID men than for the treated men.

**Conclusion:** step 2 left small differences between the weighted PSID men and the treated men, and guessed that these differences make weighting's estimate a bit off. Valued with the earnings formula, the differences are worth $294.0 of 1978 earnings. The $294.0 prices each leftover column difference with the earnings formula's coefficients. But the weighted mean miss, (−$1,262.3), shows the earnings formula fits men like the treated men badly, so the earnings formula's coefficients may be wrong for them too. So the $294.0 is only a rough measure.

## Exercise — weighting and doubly robust with the CPS men

Redo both estimates with a different comparison group: the 15,992 CPS (Current Population Survey, a monthly US labour survey) men, instead of the PSID men. Fit the scores with the same 8 columns, and the earnings formula on the CPS men.

Write your answers first, then read the answers below.

### Questions

1. Where does the weight go: how many CPS men carry half the total weight, and what is the largest weight? How does that compare with the PSID men?
2. The balance check: does weighting pass the size test on all 8 columns?
3. What is the weighting estimate, its 95% confidence interval, and its gap to the benchmark, $1,794.3?
4. What is the earnings formula's weighted mean miss on the CPS men, and the doubly robust estimate, its 95% confidence interval and its gap to the benchmark?

### Answers

**Answer 1.**
- 135 of the 15,992 CPS men carry half the total weight. The largest weight is 0.955: no CPS man stands in for even one treated man.
- With the PSID men: 14 men carried half the weight, and the largest weight was 10.8.
- Why: weight = score ÷ (1 − score). Turned around: weight × (1 − score) = score → weight − (weight × score) = score → weight = score + (weight × score) = score × (1 + weight) → score = weight ÷ (1 + weight). A weight of 0.955 means a score of 0.955 ÷ 1.955 = 0.488 → no CPS man has a score above 0.488. No treated man does either: in the score formula fitted with the CPS men, the highest treated man's score is 0.487. Among the 15,992 CPS men, there are enough men like each treated man that treated men are never more than about half of them. So no CPS man gets a huge weight, and the weight spreads over many men.

**Answer 2.** Yes, all 8 columns pass.

| column | type | treated men (185 men): mean | all CPS men (15,992 men): mean | CPS men, weighted: mean | difference after weighting | spread among the treated men (185 men) | size = difference ÷ spread | passes (size under 0.1)? |
|---|---|---|---|---|---|---|---|---|
| age | number | 25.8 | 33.2 | 25.3 | −0.5 | 7.2 | −0.069 | yes |
| years of education | number | 10.3 | 12.0 | 10.3 | 0.0 | 2.0 | 0.000 | yes |
| black | 1/0 | 84.3% | 7.4% | 84.2% | −0.1 | 36.5 | −0.003 | yes |
| hispanic | 1/0 | 5.9% | 7.2% | 6.0% | 0.1 | 23.7 | 0.004 | yes |
| married | 1/0 | 18.9% | 71.2% | 18.4% | −0.5 | 39.3 | −0.013 | yes |
| no degree | 1/0 | 70.8% | 29.6% | 71.2% | 0.4 | 45.6 | 0.009 | yes |
| 1974 earnings | number | 2,095.6 | 14,016.8 | 2,179.6 | 84.0 | 4,886.6 | 0.017 | yes |
| 1975 earnings | number | 1,532.1 | 13,650.8 | 1,568.5 | 36.4 | 3,219.3 | 0.011 | yes |

Years of education: the 0.0 is rounded. Unrounded difference = −0.025 years.

**Answer 3.**
- Weighted mean 1978 earnings, CPS men (15,992 men) = (sum of weight × 1978 earnings) ÷ (sum of the weights) = 952,005.5 ÷ 184.185 = 5,168.7.
- ATT estimate = (mean 1978 earnings, treated men, 185 men) − (weighted mean 1978 earnings, CPS men) = 6,349.1 − 5,168.7 = 1,180.4.
- 95% confidence interval (bootstrap, 1,000 draws) = (−137.0) to 2,519.8. It contains 0.
- Gap to the benchmark = (benchmark) − (ATT estimate) = 1,794.3 − 1,180.4 = 613.9.

**Answer 4.**
- Regression adjustment, fitted on the CPS men: ATT estimate = (mean actual 1978 earnings, treated men, 185 men) − (mean predicted 1978 earnings without the program, treated men) =<br>
  = 6,349.1 − 5,659.3 = 689.8.
- Weighted mean miss of the earnings formula, CPS men = (sum of weight × miss) ÷ (sum of the weights) =<br>
  = (−107,507.6) ÷ 184.185 = (−583.7): for CPS men like the treated men, the earnings formula predicts $583.7 too high.
- Doubly robust ATT estimate = (regression adjustment's estimate) − (weighted mean miss) = 689.8 − (−583.7) = 1,273.5.
- 95% confidence interval (bootstrap, 1,000 draws) = (−58.0) to 2,611.8. It contains 0.
- Gap to the benchmark = (benchmark) − (doubly robust ATT estimate) = 1,794.3 − 1,273.5 = 520.8.

**What the CPS answers show:** the weights pass the size test on all 8 columns, better than with the PSID men, and still both estimates land over $500 below the benchmark (weighting $613.9, doubly robust $520.8). There are two possible reasons, and this data can't tell them apart:
- Chance: an estimate from one sample can land this far off by chance alone. The benchmark, $1,794.3, is inside both intervals (weighting (−$137.0) to $2,519.8, doubly robust (−$58.0) to $2,611.8).
- A trait with no column: passing the balance check shows the groups are alike on the 8 columns, not on traits the data doesn't have.

So better balance doesn't promise an estimate nearer the benchmark: the PSID men failed balance on one column and landed $35.5 off, while the CPS men passed on all 8 and landed $613.9 off.

## Interview angle

> "Users who turned on feature X spend 10% more. We weighted the users without X to look like the users with X, using a propensity score, and the gap is still 10%. Is that the effect of X?"

A strong answer: not on its own. That the gap stayed at 10% means the columns used don't explain it. It's the effect of X only if both hold:
- Balance: the weighted users without X look like the users with X on every column used, by the size test. If not, part of the 10% may come from those differences.
- No confounder without a column. No data can prove this one. How keen a user is, beyond what the columns measure, is the usual trait with no column: keen users try more features and also spend more anyway.

And how sure is the 10%? If a handful of users without X carry most of the weight, the estimate rests on them, and its interval is wide.

A doubly robust estimate adds one check. Fit a spend formula on the users without X. If the weights pass the balance check, the spend formula's weighted mean miss shows how far off the spend formula is for users like those with X. If that miss is far from 0, regression adjustment's estimate can't be trusted.

## Key takeaways

- Weighting estimates the effect on the treated men by giving each comparison man a weight, (score) ÷ (1 − score): the number of treated men he stands in for. Each treated man counts once.
- The weights make the comparison men count where the treated men are, and leave out the men unlike any treated man with no cut-off chosen by hand: the 1,330 PSID men below every treated man's score carried 0.02% of the weight.
- The price: where few comparison men are like the treated men, a few carry the estimate. Half the weight rested on 14 PSID men, and the interval was wide.
- On the real data, weighting moved the naive gap, (−$15,204.8), to $1,758.8, against a benchmark of $1,794.3. Its 95% confidence interval, (−$153.3) to $3,540.9, contains 0.
- Doubly robust = (regression adjustment's estimate) − (the earnings formula's weighted mean miss on the comparison men). It's right if either the earnings formula or the score formula is right. On the PSID men, it moved regression adjustment from $790.5 to $2,052.8: if the weights are close to right, the earnings formula predicted $1,262.3 too high for men like the treated men.
- What weighting can't fix: a confounder with no column, such as motivation.

## Appendix: why doubly robust is right if either formula is right

For the treated men, write:
- A = mean actual 1978 earnings, with the program
- N = mean 1978 earnings without the program (unseen)
- P = mean predicted 1978 earnings, from the earnings formula

Then:
1. True ATT = A − N.
2. Regression adjustment's ATT = A − P = (A − N) + (N − P) = (true ATT) + (N − P).
3. Doubly robust ATT = (regression adjustment's ATT) − (weighted mean miss) = (true ATT) + (N − P) − (weighted mean miss).

So the doubly robust ATT equals the true ATT exactly when the weighted mean miss = (N − P). That one condition holds in two ways:
- **Earnings formula right:** then P = N, so (N − P) = 0. The misses average 0 too, so the weighted mean miss = 0.
- **Score formula right:** then the group distribution within the treated men has exactly the same proportions as within the weighted PSID men. So the weighted PSID men's mean miss equals the treated men's mean miss, (N − P), which doesn't have to be 0.